In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [2]:
# ── Option 2: scrape ER reports directly (more efficient) ────────────────────

i=0
def get_er_samples():
    records = []
    
    # ── Step 1: get the ER browse page with the correct URL ──────────────────
    url = "https://www.mtsamples.com/site/pages/browse.asp?type=93-Emergency+Room+Reports"
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    # ── Step 2: get all sample links containing "Emergency" in the Type param ─
    links = soup.find_all("a", href=True)
    sample_links = [
        l["href"] for l in links
        if "sample.asp" in l["href"] and "Emergency" in l["href"]
    ]
    print(f"Found {len(sample_links)} ER sample links")
    print(sample_links[:3])  # sanity check

    # ── Step 3: scrape each sample page ──────────────────────────────────────
    for link in sample_links:
        r = requests.get(link)
        s = BeautifulSoup(r.text, "html.parser")
        try:
            title   = (s.find("h1") or s.find("h2") or s.find("title"))
            title   = title.text.strip() if title else "Unknown"
            content = (
                s.find("div", {"id": "samplecontent"}) or
                s.find("div", {"class": "sample-content"}) or
                s.find("div", {"id": "content"}) or
                s.find("article") or
                s.find("main")
            )
            text = content.get_text(separator="\n").strip() if content else None
            records.append({"title": title, "text": text})
            print(f"OK: {title}")
            print(i)
            i+=0
        except Exception as e:
            print(f"Skipping {link}: {e}")
        time.sleep(0.5)

    return pd.DataFrame(records)


In [3]:
# ── 2. Load and explore ───────────────────────────────────────────────────────
df_er = get_er_samples()
print(f"\nTotal ER reports collected: {len(df_er)}")

Found 225 ER sample links
['https://www.mtsamples.com/site/pages/sample.asp?Type=93-Emergency Room Reports&Sample=1921-Abdominal Pain - Consult', 'https://www.mtsamples.com/site/pages/sample.asp?Type=93-Emergency Room Reports&Sample=2294-Abrasions & Lacerations - ER Visit', 'https://www.mtsamples.com/site/pages/sample.asp?Type=93-Emergency Room Reports&Sample=2334-Accidental Celesta Ingestion - ER Visit']
OK: Sample Name: Abdominal Pain - Consult
Medical Specialty: 
			  
				Emergency Room Reports
OK: Sample Name: Abrasions & Lacerations - ER Visit
Medical Specialty: 
			  
				Emergency Room Reports
OK: Sample Name: Accidental Celesta Ingestion - ER Visit
Medical Specialty: 
			  
				Emergency Room Reports
OK: Sample Name: Acute Inferior Myocardial Infarction
Medical Specialty: 
			  
				Emergency Room Reports
OK: Sample Name: Agitation - ER Visit
Medical Specialty: 
			  
				Emergency Room Reports
OK: Sample Name: Air Under Diaphragm - Consult
Medical Specialty: 
			  
				Emerge

### Save data in a folder to explore

In [6]:
import os
import json

# ── 1. Save as CSV ────────────────────────────────────────────────────────────
output_dir = "mtsamples_er"
os.makedirs(output_dir, exist_ok=True)

# Save the full dataframe as CSV
df_er.to_csv(os.path.join(output_dir, "er_reports.csv"), index=False)
print(f"Saved {len(df_er)} reports to {output_dir}/er_reports.csv")

# ── 2. Optionally save each report as its own .txt file ──────────────────────
txt_dir = os.path.join(output_dir, "texts")
os.makedirs(txt_dir, exist_ok=True)

for _, row in df_er.iterrows():
    # sanitize title for use as filename
    filename = row['title'].replace("/", "-").replace(" ", "_").replace(":", "") + ".txt"
    filepath = os.path.join(txt_dir, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(row['text'] or "")

print(f"Saved individual .txt files to {txt_dir}/")



Saved 225 reports to mtsamples_er/er_reports.csv
Saved individual .txt files to mtsamples_er/texts/


In [7]:
# ── 3. Explore ────────────────────────────────────────────────────────────────
# Basic stats
print(f"\nTotal reports:   {len(df_er)}")
print(f"Missing texts:   {df_er['text'].isna().sum()}")
print(f"Avg text length: {df_er['text'].str.len().mean():.0f} chars")

# Look at a random example
sample = df_er.sample(1).iloc[0]
print(f"\n── Random example ──────────────────────────────")
print(f"Title: {sample['title']}")
print(f"Text:\n{sample['text'][:1000]}...")


Total reports:   225
Missing texts:   0
Avg text length: 11510 chars

── Random example ──────────────────────────────
Title: Sample Name: Itchy Rash - ER Visit
Medical Specialty: 
			  
				Emergency Room Reports
Text:
Home


Sitemap






Contact Us


Disclaimer















          Allergy / Immunology
        







          Autopsy
        







          Bariatrics
        







          Cardiovascular / Pulmonary
        







          Chiropractic
        







          Consult - History and Phy.
        







          Cosmetic / Plastic Surgery
        







          Dentistry
        







          Dermatology
        







          Diets and Nutritions
        







          Discharge Summary
        







          Emergency Room Reports
        







          Endocrinology
        







          ENT - Otolaryngology
        







          Gastroenterology
        







          General Medicine
        







          Hemato